In [1]:
# Auxiliary Functions.
from json_repair import repair_json

def get_data(data,template, ID ='', depth = 0):
    if isinstance(data,dict):
        get_data_sub(data,template,ID,depth)
    else:
        print("input data is not a dictionary. Skipping...")
                     
    return

def get_data_sub(data,template, ID ='', depth = 0):
    for i,dat in enumerate(data):
        ID = dat
        #print(dat)
        for da in data[dat]:
            for iii,n in enumerate(template):
                if ID not in out_conv.keys():
                    out_conv[ID] = {}
                    out = handle_broken_output(da['out'][data_name[iii]],template[n])
                    out_conv[ID][n] = check_output(out)
                else:
                    if n not in out_conv[ID].keys():
                        out = handle_broken_output(da['out'][data_name[iii]],template[n])
                        out_conv[ID][n] = check_output(out)
                    else:    
                        out = check_output(handle_broken_output(da['out'][data_name[iii]],template[n]))
                        out_conv[ID][n] = sum_dicts(out_conv[ID][n],out) 
                    
    return

def handle_broken_output(in_a, template):
    in_a = repair_output(in_a)
    try:
        out = define_structure(dict(json.loads(in_a)),dict(json.loads(template)))
    except:
        return template
    return out
 
def sum_dicts(a,b):
    for i,key in enumerate(b):
        if isinstance(b[key], int):
            a[key] = a[key]+b[key]
        if isinstance(b[key], dict):
            a[key] = sum_dicts(a[key],b[key])
    return a
    
def check_output(dic):
    w = {}
    if not isinstance(dic,dict):
        try:
            dic = dict(json.loads(dic))
        except:
            print(f"Could not load {dic}")
    for key in dic:
        if isinstance(dic[key], dict):
            w[key]=check_output(dic[key])
        else:
            if dic[key] != "":
                w[key]= 1
            else:
                w[key] = 0
    return w    
    
def define_structure(in_a,template):
    if in_a.keys() != template.keys():
        return template
    out = {} 
    for key in template:
        if isinstance(template[key],dict) and isinstance(in_a[key],dict):
            out[key] = define_structure(in_a[key], template[key])
        elif not isinstance(in_a[key], dict) and not isinstance(template[key], dict):
            out[key]  = in_a[key]
        else:
            out[key] = template[key]
    return out
    
def repair_output(in_a):
    if  len(in_a) < 10:
        return ""
    if in_a.count('}') ==  in_a.count('{'):
        return in_a
    elif in_a.count('}') > in_a.count('{'):
        try:
            json.loads("{" + in_a)
            in_a = "{" + in_a
        except:
            try:
                json.loads(in_a[1:])
                in_a = in_a[1:]
            except:
                try:
                    in_a = repair_json(in_a)
                    print("Repaired Output")
                    return in_a
                except:
                    #print("Output not recoverable {")
                    return ""
                
        return in_a
    elif in_a.count('}') < in_a.count('{'):
        try:
            json.loads( in_a + "}")
            in_a =  in_a + "}"
        except:
            try:
                json.loads(in_a[:-1])
                in_a = in_a[:-1]
            except:
                #print("Output not recoverable }")
                return ""
 


In [2]:
# This process is required as the model output is given in a per prompt basis. 
#In some cases, the labels are given in a per-conversation scope.
# With this code, the model outputs are simplified to a per-conversation value.
import json
import time

dataset_name = "./outputs/vllm_test_Qwen2.5-3B-Instruct_ShareGPT_main_v1_00_00_November_21_2025.json"
template_name = "./data/template.json"
output_file = "./processed/Qwen25_3B_Presidio_ShareGPT_cap_v1.json"

# Load the template json.
with open(template_name,'r') as j:
    template_2 = json.load(j)

template = template_2['fields']
data_name = template_2['data']
template_2 = []

#Read the model output file.
with open(dataset_name, 'r') as file: 
    data_1 = json.load(file)
    
# Process the output
out_conv = {}

get_data(data_1,template)

# Save the processed output.
with open(output_file, 'w') as f:
    json.dump(out_conv, f, indent=4)